## **Task 3: Image Classifier with Teachable Machine Model**

This task successfully demonstrated the deployment and use of a pre-trained image classification model from Google's Teachable Machine (TM) within a Python environment.

Implementation Summary:
Model Creation: An image classification model was trained on the TM website using webcam images for three classes (Happy, Sad, Neutral Face).

Model Deployment Fix (TFLite): The initial deployment using the standard Keras (.h5) export failed due to a deep version incompatibility between the TM model structure and the latest Google Colab TensorFlow runtime. The professional solution was to re-export the model in the TensorFlow Lite (.tflite) format.

TFLite Deployment: The model.tflite file was loaded using the dedicated tf.lite.Interpreter, which successfully bypassed all version conflicts.

Webcam Integration: The solution used Colab-specific JavaScript integration to capture a live webcam image on demand.

Preprocessing: The captured image was preprocessed according to TM's requirements: resized to 224x224 and normalized using the (x / 127.5) - 1.0 formula.

Classification: The preprocessed data was fed into the TFLite interpreter for immediate prediction, successfully outputting the predicted class and confidence score. This fulfilled the learning goals of understanding model deployment without training from scratch.

In [ ]:
import tensorflow as tf # We use the core TF library now
import numpy as np
from PIL import Image, ImageOps
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import io

# --- CONFIGURATION (Paths are now TFLite) ---
TFLITE_MODEL_PATH = "model_unquant.tflite" # New file name
LABELS_PATH = "labels.txt"
IMAGE_SIZE = (224, 224)
# ----------------------------------------------------------------

def take_photo(filename='webcam_image.jpg', quality=0.8):
    """Function to capture a photo from the webcam specifically for Google Colab."""
    js = Javascript('''
        async function takePhoto(quality) {
          const div = document.createElement('div');
          const video = document.createElement('video');
          video.style.display = 'block';
          // ... (rest of webcam capture code remains the same)
          const stream = await navigator.mediaDevices.getUserMedia({video: true});
          document.body.appendChild(div);
          div.appendChild(video);
          video.srcObject = stream;
          await video.play();
          const canvas = document.createElement('canvas');
          canvas.width = 224;
          canvas.height = 224;
          canvas.getContext('2d').drawImage(video, 0, 0, 224, 224);
          stream.getVideoTracks()[0].stop();
          div.remove();
          return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename


def classify_tflite_image():
    """Main function: Loads the TFLite model, captures the image, and runs classification."""
    print("--- Starting TFLite Image Classifier (Final Solution) ---")
    np.set_printoptions(suppress=True)

    # STEP 1: Load TFLite Model (No more Keras version issues!)
    try:
        interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH)
        interpreter.allocate_tensors()
        input_details = interpreter.get_input_details()
        output_details = interpreter.get_output_details()
        print("✅ TFLite Model loaded successfully.")
    except Exception as e:
        print(f"❌ CRITICAL ERROR: Could not load TFLite model. Error: {e}")
        print("Ensure 'model.tflite' is uploaded.")
        return

    # Load the labels
    try:
        class_names = open(LABELS_PATH, "r").readlines()
    except FileNotFoundError:
        print(f"❌ CRITICAL ERROR: Label file '{LABELS_PATH}' not found.")
        return

    # STEP 2: Capture Image
    print("\n📸 ACTION REQUIRED: Please allow webcam access and pose for the photo.")
    try:
        image_path = take_photo()
        print(f"Captured image saved as {image_path}")
    except Exception as e:
        print(f"❌ ERROR: Webcam capture failed. Error: {e}")
        return

    # STEP 3: Preprocess Image (TFLite requires Float32 and 1,224,224,3 shape)
    input_shape = input_details[0]['shape']
    data = np.ndarray(shape=input_shape, dtype=np.float32)

    image = Image.open(image_path).convert("RGB")
    image = ImageOps.fit(image, IMAGE_SIZE, Image.Resampling.LANCZOS)
    image_array = np.asarray(image)

    # Normalize the image (TM required formula)
    normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1
    data[0] = normalized_image_array

    # STEP 4: Predict and Output
    interpreter.set_tensor(input_details[0]['index'], data)
    interpreter.invoke()
    prediction = interpreter.get_tensor(output_details[0]['index'])

    index = np.argmax(prediction)

    class_name = class_names[index].strip()[2:]
    confidence_score = prediction[0][index]

    print("\n===================================")
    print("      CLASSIFICATION RESULT      ")
    print("===================================")
    print(f"Predicted Class: **{class_name}**")
    print(f"Confidence: {confidence_score*100:.2f}%")
    print("-----------------------------------")

    print("All Probabilities:")
    for i, name in enumerate(class_names):
        print(f" - {name.strip()[2:]}: {prediction[0][i]*100:.2f}%")

if __name__ == '__main__':
    classify_tflite_image()

--- Starting TFLite Image Classifier (Final Solution) ---
✅ TFLite Model loaded successfully.

📸 ACTION REQUIRED: Please allow webcam access and pose for the photo.


<IPython.core.display.Javascript object>

Captured image saved as webcam_image.jpg

      CLASSIFICATION RESULT      
Predicted Class: **Happy Face**
Confidence: 72.15%
-----------------------------------
All Probabilities:
 - Happy Face: 72.15%
 - Sad Face: 16.05%
 - Neutral: 11.81%
